# Desfoque Gaussiano Seletivo em Fotos de Obras

Este notebook aplica **desfoque Gaussiano** apenas nas imagens indicadas pelo relatório `relatorio_preprocessamento.csv`.

A lógica é:

1. Montar o Google Drive.
2. Ler o relatório CSV de pré-processamento.
3. Identificar as imagens que precisam de tratamento, considerando colunas como:
   - `Redução de Ruído`
   - `Remoção de Artefatos`
   - `Subtração de Fundo`
4. Aplicar desfoque Gaussiano com kernel ajustado automaticamente pelo nível de ruído.
5. Salvar:
   - imagem com desfoque;
   - imagem comparativa;
   - relatório CSV;
   - resumo em TXT.

**Autora:** Adriana Rolim  
**Versão:** 2.0 — Processamento seletivo com CSV  
**Data base:** Agosto de 2025


## 1. Importação das bibliotecas e montagem do Google Drive

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')


## 2. Configurações principais

Ajuste os caminhos abaixo, se necessário.

In [ ]:
# ============================
# CONFIGURAÇÕES DO PROJETO
# ============================

PASTA_BASE = '/content/drive/MyDrive/Python_VC'

CSV_RELATORIO = os.path.join(PASTA_BASE, "relatorio_preprocessamento.csv")
PASTA_ENTRADA = os.path.join(PASTA_BASE, "fotos_obra")
PASTA_SAIDA = os.path.join(PASTA_BASE, "output")

# Tamanho padrão do kernel, caso você queira usar valor fixo em alguma adaptação futura.
# Neste notebook, o kernel é calculado automaticamente em função do ruído detectado.
KERNEL_GAUSSIANO = (9, 9)


## 3. Verificação da configuração

Esta etapa confere se a pasta base, o CSV e a pasta de entrada existem antes de iniciar o processamento.

In [ ]:
def verificar_configuracao():
    """Verifica se todos os arquivos e pastas necessários existem."""
    print("🔍 VERIFICANDO CONFIGURAÇÃO:")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📊 Arquivo CSV: {CSV_RELATORIO}")
    print(f"📁 Pasta entrada: {PASTA_ENTRADA}")
    print(f"📁 Pasta saída: {PASTA_SAIDA}")
    print(f"🎛️ Kernel Gaussiano padrão: {KERNEL_GAUSSIANO}")

    problemas = []

    if not os.path.exists(PASTA_BASE):
        problemas.append("❌ Pasta base não encontrada!")

    if not os.path.exists(CSV_RELATORIO):
        problemas.append("❌ Arquivo CSV de relatório não encontrado!")
        problemas.append("💡 Execute primeiro o script de análise exploratória.")

    if not os.path.exists(PASTA_ENTRADA):
        problemas.append("❌ Pasta de entrada 'fotos_obra' não encontrada!")

    if problemas:
        print("\n".join(problemas))
        return False

    print("✅ Configuração verificada com sucesso.")
    return True


def garantir_pasta(pasta):
    """Cria a pasta se ela não existir."""
    os.makedirs(pasta, exist_ok=True)
    print(f"📁 Pasta garantida: {pasta}")


## 4. Leitura do CSV e seleção das imagens

Esta função identifica quais imagens precisam de desfoque Gaussiano.  
Ela considera as colunas técnicas relacionadas à necessidade de suavização da imagem.

In [ ]:
def converter_coluna_para_bool(serie):
    """
    Converte uma coluna para booleano de forma robusta.

    Aceita valores como:
    - True / False
    - TRUE / FALSE
    - Sim / Não
    - 1 / 0
    """
    if serie.dtype == bool:
        return serie

    return (
        serie.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "sim": True,
            "não": False,
            "nao": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False
        })
        .fillna(False)
        .astype(bool)
    )


def ler_imagens_para_desfoque(csv_path):
    """
    Lê o CSV e identifica quais imagens precisam de desfoque Gaussiano.

    Considera imagens que precisam de:
    - Redução de Ruído
    - Remoção de Artefatos
    - Subtração de Fundo

    Parâmetros:
        csv_path (str): caminho para o arquivo CSV

    Retorno:
        list: lista de nomes de arquivos que precisam de desfoque
    """
    print("\n📊 LENDO RELATÓRIO CSV PARA DESFOQUE GAUSSIANO...")

    try:
        df = pd.read_csv(csv_path)
        print(f"✅ CSV carregado com {len(df)} imagens analisadas")

        if "Imagem" not in df.columns:
            print("❌ Coluna 'Imagem' não encontrada no CSV.")
            print("📋 Colunas disponíveis:", list(df.columns))
            return []

        colunas_desfoque = [
            "Redução de Ruído",
            "Remoção de Artefatos",
            "Subtração de Fundo"
        ]

        colunas_encontradas = [col for col in colunas_desfoque if col in df.columns]

        if not colunas_encontradas:
            print("❌ Nenhuma coluna relevante para desfoque foi encontrada no CSV.")
            print("📋 Colunas disponíveis:", list(df.columns))
            return []

        condicao_desfoque = pd.Series(False, index=df.index)

        for col in colunas_encontradas:
            condicao_desfoque = condicao_desfoque | converter_coluna_para_bool(df[col])

        imagens_para_processar = df.loc[condicao_desfoque, "Imagem"].dropna().astype(str).tolist()

        print("\n📈 Estatísticas do CSV para desfoque:")
        print(f"   - Total de imagens analisadas: {len(df)}")
        print(f"   - Precisam de desfoque Gaussiano: {len(imagens_para_processar)}")
        print(f"   - Colunas consideradas: {colunas_encontradas}")

        if imagens_para_processar:
            print("\n📝 Primeiras imagens que serão processadas com desfoque:")
            for img in imagens_para_processar[:5]:
                print(f"   ✅ {img}")

            if len(imagens_para_processar) > 5:
                print(f"   ... e mais {len(imagens_para_processar) - 5} imagens")
        else:
            print("ℹ️ Nenhuma imagem precisa de desfoque Gaussiano segundo o CSV.")

        return imagens_para_processar

    except Exception as e:
        print(f"❌ Erro ao ler o CSV: {str(e)}")
        return []


## 5. Cálculo automático do nível de desfoque

O kernel é ajustado automaticamente com base no desvio padrão da imagem em escala de cinza:

- ruído alto: kernel `(15, 15)`;
- ruído médio: kernel `(11, 11)`;
- ruído baixo: kernel `(7, 7)`.

In [ ]:
def calcular_nivel_desfoque(imagem):
    """
    Calcula o nível de desfoque apropriado com base nas características da imagem.

    Parâmetros:
        imagem: imagem OpenCV em BGR

    Retorno:
        tuple: kernel_size, nivel, desvio_padrao
    """
    cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    desvio_padrao = np.std(cinza)

    if desvio_padrao > 60:
        kernel_size = (15, 15)
        nivel = "Alto"
    elif desvio_padrao > 40:
        kernel_size = (11, 11)
        nivel = "Médio"
    else:
        kernel_size = (7, 7)
        nivel = "Baixo"

    return kernel_size, nivel, desvio_padrao


def aplicar_desfoque_gaussiano(imagem, nome_arquivo, kernel_size=None):
    """
    Aplica desfoque Gaussiano na imagem.

    Parâmetros:
        imagem: imagem OpenCV em BGR
        nome_arquivo: nome do arquivo para registro
        kernel_size: tamanho do kernel. Se None, calcula automaticamente.

    Retorno:
        tuple: imagem_desfocada_rgb, parametros
    """
    print(f"   🎛️ Processando: {nome_arquivo}")

    imagem_rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

    if kernel_size is None:
        kernel_size, nivel_ruido, desvio_padrao = calcular_nivel_desfoque(imagem)
        print(f"   📊 Ruído detectado: {desvio_padrao:.2f} — nível {nivel_ruido}")
    else:
        nivel_ruido = "Fixo"
        desvio_padrao = np.std(cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY))

    desfoque = cv2.GaussianBlur(imagem_rgb, kernel_size, 0)

    parametros = {
        "kernel_size": kernel_size,
        "nivel_ruido": nivel_ruido,
        "desvio_padrao": desvio_padrao,
        "dimensoes_originais": imagem.shape[:2]
    }

    return desfoque, parametros


## 6. Salvamento das imagens processadas

Para cada imagem processada, o notebook gera:

- uma imagem com desfoque;
- uma imagem comparativa com original e resultado lado a lado.

In [ ]:
def salvar_resultados_desfoque(imagem_original, imagem_desfocada, parametros, nome_arquivo, pasta_saida):
    """
    Salva os resultados do desfoque Gaussiano.

    Parâmetros:
        imagem_original: imagem original em BGR
        imagem_desfocada: imagem com desfoque em RGB
        parametros: dicionário com parâmetros usados
        nome_arquivo: nome do arquivo original
        pasta_saida: pasta para salvar os resultados
    """
    nome_base = os.path.splitext(nome_arquivo)[0]

    # Converter de RGB para BGR para salvar corretamente com OpenCV
    desfoque_bgr = cv2.cvtColor(imagem_desfocada, cv2.COLOR_RGB2BGR)

    caminho_desfoque = os.path.join(pasta_saida, f"{nome_base}_desfoque.png")
    cv2.imwrite(caminho_desfoque, desfoque_bgr)

    # Criar visualização comparativa
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    axes[0].imshow(cv2.cvtColor(imagem_original, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Imagem Original")
    axes[0].axis("off")

    axes[1].imshow(imagem_desfocada)
    axes[1].set_title(f"Desfoque Gaussiano\nKernel: {parametros['kernel_size']}")
    axes[1].axis("off")

    plt.tight_layout()

    caminho_comparativo = os.path.join(pasta_saida, f"{nome_base}_comparativo.png")
    plt.savefig(caminho_comparativo, dpi=150, bbox_inches="tight")
    plt.close()

    return caminho_desfoque, caminho_comparativo


## 7. Processamento em lote

Esta etapa percorre apenas as imagens indicadas pelo CSV e aplica o desfoque Gaussiano.

In [ ]:
def processar_desfoque_em_lote(pasta_entrada, pasta_saida, lista_imagens):
    """
    Processa as imagens em lote aplicando desfoque Gaussiano.

    Parâmetros:
        pasta_entrada: pasta com imagens originais
        pasta_saida: pasta para salvar resultados
        lista_imagens: lista de imagens para processar

    Retorno:
        list: lista de dicionários com os resultados por imagem
    """
    garantir_pasta(pasta_saida)

    if not lista_imagens:
        print("ℹ️ Nenhuma imagem para processar.")
        return []

    print(f"\n🔄 PROCESSANDO {len(lista_imagens)} IMAGENS COM DESFOQUE GAUSSIANO...")
    print("=" * 60)

    resultados_gerais = []
    processadas = 0
    nao_encontradas = 0
    erros = 0

    for arquivo in lista_imagens:
        caminho_entrada = os.path.join(pasta_entrada, arquivo)

        if not os.path.exists(caminho_entrada):
            print(f"❌ Imagem não encontrada: {arquivo}")
            nao_encontradas += 1
            continue

        try:
            imagem_original = cv2.imread(caminho_entrada)

            if imagem_original is None:
                print(f"❌ Não foi possível abrir: {arquivo}")
                erros += 1
                continue

            imagem_desfocada, parametros = aplicar_desfoque_gaussiano(
                imagem_original,
                arquivo
            )

            caminho_desfoque, caminho_comparativo = salvar_resultados_desfoque(
                imagem_original,
                imagem_desfocada,
                parametros,
                arquivo,
                pasta_saida
            )

            resultados_gerais.append({
                "imagem": arquivo,
                "kernel_size": parametros["kernel_size"],
                "nivel_ruido": parametros["nivel_ruido"],
                "desvio_padrao": parametros["desvio_padrao"],
                "dimensoes_originais": parametros["dimensoes_originais"],
                "caminho_desfoque": caminho_desfoque,
                "caminho_comparativo": caminho_comparativo
            })

            processadas += 1
            print(f"   ✅ Desfoque aplicado: {arquivo} — kernel: {parametros['kernel_size']}")

        except Exception as e:
            print(f"❌ Erro ao processar {arquivo}: {str(e)}")
            erros += 1

    print("\n" + "=" * 60)
    print("📊 RESUMO DO DESFOQUE GAUSSIANO:")
    print(f"   ✅ Imagens processadas com sucesso: {processadas}")
    print(f"   ❌ Imagens não encontradas: {nao_encontradas}")
    print(f"   ❌ Erros de processamento: {erros}")
    print(f"   📁 Pasta de saída: {pasta_saida}")

    if resultados_gerais:
        kernels_utilizados = [r["kernel_size"] for r in resultados_gerais]
        kernel_mais_comum = max(set(kernels_utilizados), key=kernels_utilizados.count)
        print(f"   🎛️ Kernel mais utilizado: {kernel_mais_comum}")

    return resultados_gerais


## 8. Relatórios e visualização de exemplos

O notebook gera um CSV com os parâmetros aplicados e um resumo textual do processamento.

In [ ]:
def gerar_relatorio_desfoque(pasta_saida, resultados):
    """Gera relatório detalhado do desfoque aplicado."""
    if not resultados:
        print("ℹ️ Nenhum resultado para gerar relatório.")
        return

    relatorio_path = os.path.join(pasta_saida, "relatorio_desfoque.csv")

    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(relatorio_path, index=False)

    print(f"📄 Relatório de desfoque salvo: {relatorio_path}")

    resumo_path = os.path.join(pasta_saida, "resumo_desfoque.txt")

    with open(resumo_path, "w", encoding="utf-8") as f:
        f.write("RELATÓRIO DE DESFOQUE GAUSSIANO\n")
        f.write("=" * 50 + "\n")
        f.write(f"Data: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total de imagens processadas: {len(resultados)}\n")

        kernels = [r["kernel_size"] for r in resultados]
        kernel_counts = {k: kernels.count(k) for k in set(kernels)}

        f.write("\nDISTRIBUIÇÃO DOS KERNELS:\n")
        for kernel, count in kernel_counts.items():
            f.write(f"  Kernel {kernel}: {count} imagens\n")

        niveis = [r["nivel_ruido"] for r in resultados]
        nivel_counts = {n: niveis.count(n) for n in set(niveis)}

        f.write("\nNÍVEIS DE RUÍDO DETECTADOS:\n")
        for nivel, count in nivel_counts.items():
            f.write(f"  {nivel}: {count} imagens\n")

        f.write("\nDETALHES POR IMAGEM:\n")
        for res in resultados:
            f.write(
                f"- {res['imagem']}: "
                f"Kernel {res['kernel_size']}, "
                f"Ruído {res['nivel_ruido']}, "
                f"Desvio padrão {res['desvio_padrao']:.2f}\n"
            )

    print(f"📋 Resumo de desfoque salvo: {resumo_path}")


def mostrar_exemplo_resultados(pasta_saida, num_exemplos=3):
    """Mostra exemplos dos resultados comparativos gerados."""
    if not os.path.exists(pasta_saida):
        print("ℹ️ Pasta de saída ainda não existe.")
        return

    arquivos_comparativos = [
        f for f in os.listdir(pasta_saida)
        if f.endswith("_comparativo.png")
    ]

    if not arquivos_comparativos:
        print("ℹ️ Nenhum arquivo comparativo encontrado para exibir.")
        return

    exemplos = arquivos_comparativos[:num_exemplos]

    print(f"\n🖼️ EXIBINDO {len(exemplos)} EXEMPLOS DE RESULTADOS:")

    for exemplo in exemplos:
        caminho_exemplo = os.path.join(pasta_saida, exemplo)
        img = plt.imread(caminho_exemplo)

        plt.figure(figsize=(12, 5))
        plt.imshow(img)
        plt.title(f"Exemplo: {exemplo}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()


## 9. Execução do processamento

Execute a célula abaixo para iniciar o fluxo completo.

In [ ]:
def executar_processamento_desfoque():
    print("🚀 INICIANDO APLICAÇÃO DE DESFOQUE GAUSSIANO — PROCESSAMENTO SELETIVO")
    print("=" * 70)

    if not verificar_configuracao():
        print("\n❌ Não foi possível iniciar o processamento de desfoque.")
        print("\n💡 SOLUÇÃO:")
        print("1. Execute primeiro o script de análise exploratória.")
        print("2. Certifique-se de que o arquivo 'relatorio_preprocessamento.csv' foi gerado.")
        print("3. Verifique se a pasta 'fotos_obra' contém as imagens originais.")
        return []

    imagens_para_processar = ler_imagens_para_desfoque(CSV_RELATORIO)

    if not imagens_para_processar:
        print("\nℹ️ Nenhuma imagem necessita de desfoque Gaussiano.")
        print("💡 Todas as imagens já têm qualidade adequada segundo o CSV.")
        return []

    resultados = processar_desfoque_em_lote(
        PASTA_ENTRADA,
        PASTA_SAIDA,
        imagens_para_processar
    )

    if resultados:
        gerar_relatorio_desfoque(PASTA_SAIDA, resultados)

        print("\n🎯 DESFOQUE GAUSSIANO CONCLUÍDO!")
        print(f"   📁 Resultados salvos em: {PASTA_SAIDA}")
        print(f"   📊 {len(resultados)} imagens processadas com sucesso")
        print("   🎛️ Níveis de desfoque ajustados automaticamente ao ruído de cada imagem")

        mostrar_exemplo_resultados(PASTA_SAIDA)

    return resultados


resultados_desfoque = executar_processamento_desfoque()
